# 📖 中文小说模型对比测试 · AMD ROCm 主路线

在 ModelScope 免费 AMD GPU 实例（192GB 显存 · ROCm）上测试中文小说模型。当前第一优先：**玄幻 DPO**，以**最接近作者训练条件**的方式运行——BF16（未量化）底模 + LoRA，不引入量化误差，测的是模型本身而不是"模型+4bit"。

**怎么用（就这么简单）：**
1. 创建实例时选 **AMD GPU 镜像**（推荐 `ubuntu22.04-rocm7.2.3-py312-torch2.11.0-1.39.0`）；
2. 菜单 **Run → Run All Cells（运行全部）**；
3. 第 5 步自动下载并加载模型②（首次约 56.6GB，之后从持久盘秒加载），加载完**自动跑 Smoke Test** 自检；
4. 第 6 步在输入框粘贴剧情（官方 〖〗结构示例已预填）→ 点【✍️ 开始生成】→ 正文逐字显示并自动保存。

| # | 模型 | 运行方式 | 下载体积 | 当前状态 |
|---|------|---------|---------|----------|
| ② | **玄幻 DPO（第一优先）** | Qwen3.8-27B **BF16（未量化）** + LoRA · Transformers/PEFT | ≈56.6GB | ✅ 本分支主路线（AMD ROCm） |
| ① | WebNovel Writer | llama.cpp + Q4_K_M | ≈16.5GB | NVIDIA 历史路线，AMD 适配属第二阶段 |

> ⚠️ **为什么用 BF16 不用 4bit**：我们要判断"AI 味有没有减少"这类细微差别，就不能在第一轮主动引入量化这个新变量；且 192GB 显存下完全不需要省。模型卡作者的官方用法也是不量化推理。
>
> 💡 **省时长**：GPU 时长按开机时间计（下载也计时）。想省额度，可先在免费不限时的 CPU 实例上 Run All 预下载模型（Notebook 会自动识别并只下载不生成），再开 AMD 实例直接加载。
>
> 遇到红色报错不要慌，截图按 README「常见问题」处理。

## 第 1 步 · 配置（整个项目唯一可能需要看的代码格）

默认值已按模型作者的官方说明填好——**直接用，什么都不用改**。所有模型 ID、存储路径和生成参数都集中在这里；其他 Cell 都是自动流程，请勿修改。

In [ ]:
# ============================================================
# 第 1 步：全局配置（唯一需要看的 Cell，默认值已按模型作者官方说明填好）
# ============================================================
CONFIG = {
    # ---------- 存储位置（ModelScope 免费实例自带 100GB 持久盘，关机不丢） ----------
    "workspace_root": "/mnt/workspace",
    # 模型权重 + 下载缓存统一放这里（只保留一份，避免双份缓存吃掉磁盘）
    "models_root": "/mnt/workspace/models",
    # Hugging Face 下载缓存目录（与模型同盘）
    "hf_cache": "/mnt/workspace/models/hf-cache",
    # 生成结果保存目录
    "results_dir": "/mnt/workspace/novel-model-benchmark/results",
    # 批量盲评模式的提示词文件夹（Notebook 同级的 prompts/）
    "prompts_dir": "prompts",

    # ---------- 下载源（按优先级自动切换） ----------
    "hf_official": "https://huggingface.co",  # 官方源，优先
    "hf_mirror":   "https://hf-mirror.com",   # 第三方镜像（非官方，仅在官方源失败时自动使用，不发送任何令牌）

    # ---------- 两个模型 ----------
    "models": {
        # 模型②（当前第一优先）：玄幻 DPO —— BF16（未量化）底模 + LoRA，Transformers/PEFT 运行
        "xuanhuan": {
            "label": "② 玄幻 DPO · BF16（未量化）+ LoRA（约56.6GB，AMD 主路线）",
            "type": "peft_bf16",
            "adapter_repo": "JiangLing-js/Qwen3.8-27B-Chinese-Xuanhuan-Novel-Writer-DPO",
            "base_repo": "unsloth/Qwen3.8-27B",      # BF16 原始权重（约55.6GB）
            "context": 4096,
            "max_prompt_tokens": 2800,               # 作者训练分布：max_prompt 2800 / max_completion 1900
            # ↓ 生成参数 = 模型作者 held-out 测试原参数
            "temperature": 0.85, "top_p": 0.90, "top_k": 40, "repetition_penalty": 1.05,
            # ↓ System prompt = 作者模型卡示例原文
            "system_prompt": "你是一名中文长篇小说写作助手，只输出小说正文。",
            "max_new_tokens": 1700,   # 生成上限（防写不停）；实际篇幅由提示词中的字数目标控制
            "disable_thinking": True, # 训练与 held-out 测试均关闭思考模式，推理必须一致
            "min_disk_gb": 70,        # 下载前建议的磁盘余量（55.6GB 底模 + 缓存余量）
        },
        # 模型①（第二优先）：西幻网文写作模型 —— llama.cpp 路线（面向 NVIDIA，AMD 适配属第二阶段）
        "webnovel": {
            "label": "① WebNovel Writer · 西幻白描风（llama.cpp/NVIDIA 路线，约16.5GB）",
            "type": "gguf",
            "repo_id": "wcn123/Qwen3.5-27B-WebNovel-Writer-zh-GGUF",
            "gguf_file": "Qwen3.5-27B-WebNovel-Writer-zh-Q4_K_M.gguf",
            "quant": "Q4_K_M",          # 仓库内唯一量化版（NVIDIA 路线专用）
            "context": 4096,
            # ↓ 生成参数 = 模型卡「使用建议」官方推荐值
            "temperature": 0.7, "top_p": 0.90, "repeat_penalty": 1.10,
            "system_prompt": "你是一位中文西幻网文写作助手，擅长创作高质量的小说正文。请根据用户的指令完成写作任务。",
            "max_new_tokens": 1700,
        },
    },
    "default_model": "xuanhuan",  # 「运行全部」时自动加载模型②（当前第一优先）

    # ---------- 通用 ----------
    "target_chars": "500~1000",   # 提示词中声明的正文篇幅目标（中文字）
}

# 环境变量必须在 import huggingface 之前设置：把缓存统一指到持久盘，只留一份权重
import os
os.environ["HF_HOME"] = CONFIG["hf_cache"]
for d in (CONFIG["models_root"], CONFIG["results_dir"]):
    os.makedirs(d, exist_ok=True)

SHORT_NAME = {"webnovel": "webnovel-writer", "xuanhuan": "xuanhuan-dpo"}

print("✅ 配置加载完成")
print(f"   模型缓存目录 : {CONFIG['models_root']}")
print(f"   结果保存目录 : {CONFIG['results_dir']}")
print(f"   默认模型     : {CONFIG['default_model']}（{CONFIG['models'][CONFIG['default_model']]['label']}）")

## 第 2 步 · 环境自检（自动）

用 PyTorch 识别 GPU（AMD ROCm / NVIDIA CUDA 都认），不再依赖 `nvidia-smi`。下载大文件前先确认显存、磁盘、内存够用；没有 GPU 时自动进入「CPU 预下载模式」（只下载、不生成，不占 GPU 时长）。

In [ ]:
# ============================================================
# 第 2 步：环境自检
#   GPU 识别以 PyTorch 为准：torch.cuda 在 AMD(ROCm/HIP) 和 NVIDIA(CUDA) 上都能用；
#   AMD 与 NVIDIA 靠 torch.version.hip / torch.version.cuda 区分。
#   nvidia-smi / rocm-smi 只作辅助显示，绝不作为 GPU 存在与否的唯一判断。
# ============================================================
import platform, shutil, subprocess

def _run(cmd):
    """执行 shell 命令，返回 stdout（失败返回空字符串，不抛异常）。"""
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception:
        return ""

problems = []
try:
    import torch
except Exception:
    torch = None

GPU_KIND = None      # "amd" / "nvidia" / "other" / None
GPU_INFO = {}
if torch is not None and torch.cuda.is_available():
    # PyTorch 的 HIP 兼容层：AMD GPU 也走 torch.cuda API
    GPU_INFO["name"] = torch.cuda.get_device_name(0)
    GPU_INFO["vram"] = torch.cuda.get_device_properties(0).total_memory / 1024**3
    GPU_INFO["torch"] = torch.__version__
    if getattr(torch.version, "hip", None):
        GPU_KIND = "amd"
        GPU_INFO["hip"] = torch.version.hip
    elif getattr(torch.version, "cuda", None):
        GPU_KIND = "nvidia"
        GPU_INFO["cuda"] = torch.version.cuda
    else:
        GPU_KIND = "other"
else:
    # torch 不可用时用系统工具兜底探测
    for cmd, kind in (("amd-smi", "amd"), ("rocm-smi", "amd"), ("nvidia-smi -L", "nvidia")):
        if _run(cmd).strip():
            GPU_KIND = kind
            GPU_INFO["note"] = "（torch 不可用，由系统工具探测）"
            break

NO_GPU = GPU_KIND is None

if GPU_KIND == "amd":
    print(f"✅ GPU：{GPU_INFO['name']}｜显存 {GPU_INFO['vram']:.0f} GiB｜ROCm/HIP {GPU_INFO.get('hip')}｜PyTorch {GPU_INFO['torch']}｜Python {platform.python_version()}")
    if GPU_INFO["vram"] >= 80:
        print("   🎯 推荐环境：显存充裕，玄幻 DPO 将以 BF16 全精度直接运行（最接近作者训练/测试条件）。")
    else:
        print("   ⚠️ 显存不足 80GiB：BF16 全精度（约56GB）会很紧张，建议换 192GB 显存的 AMD 实例。")
        if GPU_INFO["vram"] < 60:
            problems.append("显存不足以BF16加载")
elif GPU_KIND == "nvidia":
    print(f"✅ GPU：{GPU_INFO.get('name','?')}｜显存 {GPU_INFO.get('vram',0):.0f} GiB｜CUDA {GPU_INFO.get('cuda','?')}｜PyTorch {GPU_INFO.get('torch','?')}")
    print("   ℹ️  当前主路线为 AMD ROCm BF16；NVIDIA 环境下模型②请参考 main 分支的 A10/4bit 历史方案。")
    if GPU_INFO.get("vram", 0) < 22:
        print("   ⚠️ 显存不足 22GiB：任何 27B 路线都跑不动。")
        problems.append("显存不足")
elif GPU_KIND == "other":
    print(f"⚠️  检测到 GPU 但无法判断加速后端（{GPU_INFO.get('name','?')}），继续运行，遇到问题截图求助。")
else:
    print("⚠️  没有检测到 GPU —— 当前是 CPU 环境（ModelScope CPU 实例免费、时长不限）。")
    print("   CPU 环境不能生成正文，但适合【预下载】模型：下载完切换/启动 AMD GPU 实例即可加载，")
    print("   下载时间完全不占用 GPU 时长。继续运行下面的 Cell 即可自动下载。")

# ---------- 磁盘：BF16 底模 55.6GB + LoRA + 缓存余量 ----------
if os.path.exists(CONFIG["workspace_root"]):
    total, used, free = shutil.disk_usage(CONFIG["workspace_root"])
    print(f"✅ 磁盘：{CONFIG['workspace_root']} 剩余 {free/1000**3:.0f}GB / 共 {total/1000**3:.0f}GB")
    need = CONFIG["models"]["xuanhuan"]["min_disk_gb"]
    if free < 58 * 1000**3:
        print(f"   🔴 剩余不足 58GB：装不下 BF16 底模（55.6GB），请先清理 /mnt/workspace。")
        problems.append("磁盘不足")
    elif free < need * 1000**3:
        print(f"   ⚠️ 剩余不足建议值 {need}GB：能装下但比较紧，建议清理后再开始。")
else:
    print(f"⚠️ 未找到 {CONFIG['workspace_root']}（本机不是 ModelScope 环境？云端会自动出现）")

# ---------- 内存 ----------
freeg = _run("free -g")
try:
    avail = int([l for l in freeg.splitlines() if l.startswith("Mem:")][0].split()[6])
    print(f"✅ 内存：可用约 {avail}GB")
except Exception:
    pass

if problems:
    print(f"\n🔴 自检未通过（{problems}）。请先按上面的提示解决，再重新运行本 Cell。")
    raise SystemExit("环境自检未通过")
if NO_GPU:
    print("\n🟡 CPU 预下载模式已就绪：继续运行下面的 Cell 会自动下载玄幻 DPO 所需文件。")
else:
    print("\n🟢 环境自检通过，请继续运行下一个 Cell。")

## 第 3 步 · 自动安装基础依赖（1~3 分钟）

只装缺失的轻量通用包（下载器、交互控件），失败自动换清华镜像源。**硬性禁令**：任何时候都不安装/覆盖 `torch / torchvision / torchaudio / vllm`——ModelScope 的 ROCm + PyTorch 是平台预装并适配好的，动了会弄坏 GPU 环境。深度学习上层包（transformers/peft 等）等第 5 步加载模型时按需补装并做前后校验。

In [ ]:
# ============================================================
# 第 3 步：依赖自动安装（缺什么装什么；默认源失败自动换清华镜像）
#   ⛔ 硬性禁令：绝不安装 torch/torchvision/torchaudio/vllm（保护平台 ROCm 环境）
# ============================================================
import re, importlib, subprocess, sys

def _pkg_name(p):
    """从 pip 规格串里提取包名：'transformers>=5.3' → 'transformers'。"""
    m = re.match(r"[A-Za-z0-9_.\-]+", p)
    return m.group(0) if m else p

def pip_install(pkgs):
    """安装缺失的 pip 包（已装且无版本要求的跳过）；默认源失败自动换清华镜像。"""
    need = []
    for p in pkgs:
        try:
            importlib.import_module(_pkg_name(p).replace("-", "_"))
        except ImportError:
            need.append(p)
    if not need:
        print("✅ 依赖已就绪：" + ", ".join(_pkg_name(p) for p in pkgs))
        return
    print(f"⏳ 安装缺失依赖：{need} …")
    last_err = ""
    for extra in ([], ["-i", "https://pypi.tuna.tsinghua.edu.cn/simple"]):
        cmd = [sys.executable, "-m", "pip", "install", "-q", *extra, *need]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            break
        last_err = r.stderr[-600:]
    else:
        raise RuntimeError(f"依赖安装失败，请把下方信息截图求助：\n{last_err}")
    print("✅ 安装完成")

# 基础包：下载器 + 交互控件（transformers/peft/accelerate 在第 5 步按需补装并校验）
pip_install(["huggingface_hub", "ipywidgets"])

## 第 4 步 · 下载与通用工具（自动）

包含：模型存在性检查、**断点续传下载**（中断后重跑接着上次的进度继续）、官方源失败自动切第三方镜像、显存监控（AMD/NVIDIA 通吃）、结果自动保存（含运行环境元信息）、两个模型各自的官方提示词模板。

In [ ]:
# ============================================================
# 第 4 步：下载与通用工具
#   · smart_download : 先查模型是否存在 → 官方源下载(断点续传) → 失败自动切 hf-mirror
#   · show_gpu       : 显存监控（torch 优先，AMD/NVIDIA 通用）
#   · save_result    : 结果自动保存（文件名 = 模型名_日期时间_测试编号，附环境元信息）
#   · build_user_message : 按各模型作者官方输入模板包装你粘贴的剧情
# ============================================================
import os, re, gc, time, json, glob, shutil, datetime
from huggingface_hub import HfApi

STATE = {"loaded_model": None, "runner": None}   # 当前已加载的模型与运行方式

def _model_exists(repo_id, endpoint):
    """在指定下载源上检查模型是否存在（连不上视为不存在，自动换源）。"""
    try:
        return HfApi(endpoint=endpoint).model_info(repo_id, timeout=20) is not None
    except Exception:
        return False

def smart_download(repo_id, filename=None):
    """下载模型（带断点续传与镜像降级）：
       ① 官方源 huggingface.co（中断后重跑本函数自动续传）
       ② 第三方镜像 hf-mirror.com（非官方节点，同样支持续传；强制匿名，绝不向镜像发送令牌）
       实现说明：hub 1.x 的下载端点在 import 时固化，因此每个源用 HfApi(endpoint=...) 独立客户端。
       两者都失败才报错，并给出可手动执行的补救命令。"""
    for name, ep, token in [("官方源", CONFIG["hf_official"], None),
                            ("第三方镜像", CONFIG["hf_mirror"], False)]:
        if not _model_exists(repo_id, ep):
            print(f"   · {name}({ep}) 上未找到 {repo_id}（或网络不通），换下一个源…")
            continue
        try:
            what = filename or "整个仓库"
            print(f"⏳ 从{name}下载 {repo_id} / {what}")
            print("   （已下载过则直接复用；下载中断后重新运行本 Cell 会从断点继续）")
            api = HfApi(endpoint=ep, token=token)   # token=False：镜像强制匿名
            if filename:
                path = api.hf_hub_download(repo_id=repo_id, filename=filename)
            else:
                path = api.snapshot_download(repo_id=repo_id)
            print(f"✅ 已就绪：{path}")
            return path
        except Exception as e:
            print(f"   · {name}下载失败：{str(e)[:200]}")
    raise RuntimeError(
        f"\n🔴 两个下载源都失败了。补救方法（按顺序试）：\n"
        f"   1) 重新运行本 Cell（网络波动常常自愈，且会断点续传，不会重头下载）\n"
        f"   2) 仍失败：菜单 File → New → Terminal，执行下面这行后回来重跑：\n"
        f"        export HF_ENDPOINT={CONFIG['hf_mirror']}\n"
        f"   3) 还不行：截图报错信息求助。")

def show_gpu(tag="当前"):
    """显存监控：torch（AMD/NVIDIA 通用）优先，系统工具兜底。"""
    parts = []
    try:
        import torch
        if torch.cuda.is_available():
            parts.append(f"torch已分配/峰值保留 {torch.cuda.memory_allocated()/1024**3:.1f}/{torch.cuda.max_memory_reserved()/1024**3:.1f} GiB")
    except Exception:
        pass
    for c in ("amd-smi --brief", "rocm-smi --showmeminfo vram --csv",
              "nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader"):
        o = _run(c)
        if o.strip():
            parts.append(o.strip().splitlines()[-1][:90])
            break
    print(f"🖥️  {tag}：" + ("｜".join(parts) if parts else "GPU 信息不可用"))

def env_info():
    """收集运行环境元信息（写进结果 .md，用于日后追溯正文是在什么环境生成的）。"""
    lines = []
    try:
        import torch
        if torch.cuda.is_available():
            lines.append(f"GPU {torch.cuda.get_device_name(0)} {torch.cuda.get_device_properties(0).total_memory/1024**3:.0f}GiB")
            hip = getattr(torch.version, "hip", None)
            lines.append(("AMD ROCm/HIP " + str(hip)) if hip else ("NVIDIA CUDA " + str(torch.version.cuda)))
            lines.append(f"torch {torch.__version__}")
    except Exception:
        pass
    for pkg in ("transformers", "peft"):
        try:
            m = __import__(pkg)
            lines.append(f"{pkg} {getattr(m, '__version__', '?')}")
        except Exception:
            pass
    return "；".join(lines) or "未知"

def _repo_line(k):
    c = CONFIG["models"][k]
    if c["type"] == "gguf":
        return f"{c['repo_id']} · {c['gguf_file']}"
    return f"{c['adapter_repo']} + BF16底模 {c['base_repo']}"

def next_test_number(model_key):
    """扫描 results 目录，算出该模型的下一个测试编号（001 起）。"""
    pat = re.compile(re.escape(SHORT_NAME[model_key]) + r"_\d{8}_\d{6}_test(\d+)\.")
    nums = []
    for f in glob.glob(os.path.join(CONFIG["results_dir"], "*.txt")):
        m = pat.match(os.path.basename(f))
        if m:
            nums.append(int(m.group(1)))
    return max(nums, default=0) + 1

def save_result(model_key, task_label, user_text, gen_text, meta=None):
    """自动保存：.txt 纯正文 + .md 带元信息（模型/环境/参数/耗时/显存/提示词）。"""
    meta = meta or {}
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    n = next_test_number(model_key)
    stem = f"{SHORT_NAME[model_key]}_{ts}_test{n:03d}"
    txt_path = os.path.join(CONFIG["results_dir"], stem + ".txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(gen_text.strip() + "\n")
    c = CONFIG["models"][model_key]
    params = {k: c.get(k) for k in ("temperature", "top_p", "top_k", "repeat_penalty",
                                    "repetition_penalty", "max_new_tokens") if k in c}
    params["context"] = meta.get("context", c.get("context"))
    precision = "BF16（未量化，原生精度）" if c["type"] == "peft_bf16" else c.get("quant", "?")
    md_path = txt_path.replace(".txt", ".md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write("# 生成记录\n\n"
                f"- 模型：{_repo_line(model_key)}\n"
                f"- 运行环境：{meta.get('env') or env_info()}\n"
                f"- 精度/思考：{precision}｜thinking={'关闭' if c.get('disable_thinking', True) else '开启'}\n"
                f"- 任务类型：{task_label}\n- 测试编号：{n:03d}\n- 时间：{ts}\n"
                f"- 生成参数：{json.dumps(params, ensure_ascii=False)}\n"
                + (f"- Prompt tokens：{meta.get('prompt_tokens')}｜实际生成上限：{meta.get('actual_max_new_tokens')}\n" if meta.get("prompt_tokens") else "")
                + f"- 耗时：{meta.get('elapsed', '?')} 秒｜速度：{meta.get('speed', '?')}\n"
                f"- 显存：{meta.get('vram', '?')}｜峰值：{meta.get('vram_peak', '?')}\n\n"
                f"## 提示词\n\n```\n{user_text.strip()}\n```\n\n## 正文\n\n{gen_text.strip()}\n")
    print(f"💾 已自动保存：\n   {txt_path}\n   {md_path}")
    return txt_path, md_path

# ---------- 提示词模板（均来自模型作者官方说明） ----------
# 模型②：作者模型卡的官方输入结构
XUANHUAN_STRUCTURE = "〖人物状态〗\n\n〖前情〗\n\n〖事件骨架〗\n\n〖写作要求〗写成完整连续的小说场景。"
# Smoke Test 用的官方结构小样例
SMOKE_PROMPT_USER = ("〖人物状态〗\n主角处于警戒状态。\n\n〖前情〗\n主角刚进入陌生山谷。\n\n"
                     "〖事件骨架〗\n1. 主角观察环境。\n2. 听见异常声响。\n3. 暂不行动。\n\n"
                     "〖写作要求〗写成连续小说正文。")

# 模型①：模型卡「使用建议」里的 4 种任务格式，{text} 处填你粘贴的内容
WEBNOVEL_TASKS = {
    "骨架扩写": ("请将下面文本增强为更自然的小说正文。\n\n"
               "输入类型：事件骨架\n增强强度：高（从纯骨架到完整场景）\n长度目标：3.5x~6.0x\n"
               "重点：从骨架扩写完整场景：叙事结构、节奏铺排、感官填充、对话还原\n"
               "要求：严格保留骨架中的全部事实、人名、地名、术语、事件顺序与结果。不新增原文没有的信息。\n\n"
               "文本：\n{text}"),
    "场景扩写": ("任务：场景扩写\n场景描述：{text}\n"
               "要求：西幻风格，注重场景感和节奏，保持人物行为合理。\n\n请写出完整的小说场景。"),
    "正文增强": ("任务：正文增强\n目标：让这段更像成熟作者写出的西幻正文\n输入类型：场景beat\n"
               "增强强度：中（从场景beat到完整场景）\n长度目标：2.0x~3.5x\n"
               "重点：节奏铺排、心理层次、环境渲染、微动作衔接\n"
               "限制：严格保留骨架中的全部事实、人名、地名、术语、事件顺序与结果。不新增原文没有的信息。\n\n"
               "素材：\n{text}"),
    "正文润色": "任务：正文润色\n要求：提升文笔、优化节奏、保留原意\n\n原文：\n{text}",
}
WEBNOVEL_DEFAULT_TASK = "骨架扩写"

def _length_goal(s):
    return s + f"\n\n（篇幅：正文约{CONFIG['target_chars']}个中文字，写完自然收束，不要凑字数，不要输出解释性文字。）"

def build_user_message(model_key, text, task=None):
    """把你粘贴的剧情，按所选模型作者官方的输入模板包装成完整提示词。"""
    text = (text or "").strip()
    if not text:
        raise ValueError("输入为空：请先把剧情粘贴到输入框里。")
    if model_key == "webnovel":
        task = task or WEBNOVEL_DEFAULT_TASK
        return _length_goal(WEBNOVEL_TASKS[task].format(text=text))
    # 模型②：已带 〖人物状态/前情/事件骨架/写作要求〗 标签则原样使用（作者官方结构），
    # 否则视为事件骨架，自动套进官方结构
    if re.search(r"〖(人物状态|前情|事件骨架|写作要求)〗", text):
        return _length_goal(text)
    filled = ("〖人物状态〗\n（未提供）\n\n〖前情〗\n（未提供）\n\n"
              "〖事件骨架〗\n" + text + "\n\n〖写作要求〗写成完整连续的小说场景。")
    return _length_goal(filled)

def clean_output(text):
    """去掉思考模式标签（模型②训练时关闭思考，这里双保险）与首尾空白。
       未闭合的 <think> 视为生成中断在思考段，只保留标签之前的内容。"""
    text = text or ""
    if "<think>" in text:
        if "</think>" not in text:
            text = text.split("<think>")[0]
        else:
            text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
            text = text.replace("<think>", "").replace("</think>", "")
    return text.strip()

# ---------- 批量盲评模式的默认提示词（prompts/ 为空时自动生成） ----------
DEFAULT_PROMPTS = {
    "01_寒潭.txt": ("宗门后山有一口百年寒潭，潭底沉着一块黑铁。杂役弟子沈砚每日来此挑水。\n"
                 "这夜他失足落水，寒气入体本该致命，怀中却发热——那块黑铁竟顺水流进他掌心。\n"
                 "沈砚在潭底憋了半炷香才爬上岸，没死，反而觉得四肢百骸有暖流游走。\n"
                 "翌日晨课，他一拳打裂了练功的石桩。执法长老盯住他手背浮现的黑色纹路。"),
    "02_战斗.txt": ("坊市拍卖会上，沈砚看中的淬体丹被内门弟子赵崂抬价截走。\n"
                 "赵崂当众羞辱杂役也配竞价，动手将沈砚掼倒在地。\n"
                 "沈砚掌心黑铁发烫，气血翻涌，他借势起身，一记崩拳打断赵崂肋骨。\n"
                 "赵崂的随从拔刀围上。拍卖师敲锤喝止，宣布坊市内禁斗，违者逐出。\n"
                 "赵崂撂话三日后演武场见生死状。人群散去，沈砚盯着自己发颤的拳头。"),
    "03_突破.txt": ("沈砚闭关第七日，气海内的暖流凝成一线，卡在炼气九层关口。\n"
                 "黑铁传音：欲破境，需以自身气血温养铁中残魂三成。\n"
                 "沈砚咬牙引气血入铁，如割肉饲鹰，几次险些昏迷。\n"
                 "第三夜铁中残魂反哺一缕先天罡气，气海关口应声而碎。\n"
                 "破入筑基，他吐出一口黑血，发现地上血迹里混着细小铁屑。"),
}

def ensure_prompts_dir():
    d = CONFIG["prompts_dir"]
    os.makedirs(d, exist_ok=True)
    files = sorted(glob.glob(os.path.join(d, "*.txt")))
    if not files:
        for name, body in DEFAULT_PROMPTS.items():
            with open(os.path.join(d, name), "w", encoding="utf-8") as f:
                f.write(body)
        print(f"📝 已生成 {len(DEFAULT_PROMPTS)} 个示例提示词到 {d}/（可自行替换/增删 .txt 文件）")
        files = sorted(glob.glob(os.path.join(d, "*.txt")))
    return files

print("✅ 工具函数就绪（下载/续传/镜像切换/显存监控/自动保存/官方提示词模板）")

## 第 5 步 · 选择模型并加载（只处理你选中的那一个）

- **「运行全部」时自动加载模型②（玄幻 DPO）**：首次下载 BF16 底模（55.6GB）+ LoRA（约 1GB），约 20~60 分钟；之后从持久盘加载（约 5~15 分钟读入显存）；
- 加载完成后**自动运行 Smoke Test**（极短自检生成），全部通过才放行正式测试；
- 依赖补装有硬门禁：只动 transformers/peft/accelerate，**绝不触碰平台 PyTorch**，装完自动校验 GPU 环境未被改变；
- 想换模型①（WebNovel Writer）：下拉框选 → 点【加载 / 切换模型】——⚠️ 模型①的 llama.cpp 路线当前面向 NVIDIA，AMD 适配属第二阶段；
- 在 CPU 实例上本步自动变成「预下载」：只下载模型②所需文件，不加载、不生成。

In [ ]:
# ============================================================
# 第 5 步：模型加载（AMD 主路线：BF16 + LoRA + Smoke Test；含平台环境保护）
# ============================================================
import os, sys, time, subprocess, gc, zipfile, tempfile, importlib, re

# ---------- 平台 PyTorch 保护 + 上层依赖补装（硬门禁） ----------
def _ver_tuple(v):
    return tuple(int(x) for x in re.findall(r"\d+", str(v))[:3])

def _torch_state():
    """平台 PyTorch 指纹：版本 + HIP 后端 + GPU 可见性。装依赖前后必须一致。"""
    import torch
    return (torch.__version__, getattr(torch.version, "hip", None), str(torch.cuda.is_available()))

def _pip_would_touch_torch(specs):
    """pip 安装计划预检（--dry-run）：计划里出现 torch 系列就否决——事前门禁，而不是装坏后再报警。"""
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "--no-input", *specs],
                           capture_output=True, text=True, timeout=300)
    except Exception:
        print("   · dry-run 预检无法执行，退回\"装后指纹校验\"兜底")
        return True   # 无法预检时放行，靠安装后的 torch 指纹复核兜底
    if r.returncode != 0:
        print("   · dry-run 预检失败（pip 版本较旧？），退回\"装后指纹校验\"兜底")
        return True
    plan = (r.stdout or "") + (r.stderr or "")
    for line in plan.splitlines():
        l = line.strip().lower()
        if ("would install" in l or "collecting" in l or "installing collected" in l):
            if re.search(r"(^|\s)(torch|torchvision|torchaudio)([-<>=~!\s]|$)", l):
                print(f"   · 预检发现安装计划包含 torch 系列：{line.strip()[:120]}")
                return False
    return True

def ensure_bench_deps():
    """只补装/升级上层包（transformers/peft/accelerate）。
       事前：pip dry-run 预检，安装计划含 torch 系列立即拒绝；
       事后：torch 指纹复核 + 版本必须真的达标。
       ⛔ 绝不安装 torch/torchvision/torchaudio/vllm——平台 ROCm+PyTorch 被覆盖会损坏 GPU 环境。"""
    try:
        pre = _torch_state()
    except Exception:
        raise SystemExit("🔴 当前环境无法导入 torch。请确认创建实例时选择了 ROCm 镜像（如 ubuntu22.04-rocm7.2.3-py312-torch2.11.0）。")
    need = []
    for pkg, minv in (("transformers", (5, 3)), ("peft", (0, 20)), ("accelerate", None)):
        try:
            m = importlib.import_module(pkg)
            cur = getattr(m, "__version__", "0")
            if minv and _ver_tuple(cur) < minv:
                need.append(f"{pkg}>={minv[0]}.{minv[1]}")
        except ImportError:
            need.append(pkg if not minv else f"{pkg}>={minv[0]}.{minv[1]}")
    if need:
        print(f"⏳ 补装/升级上层依赖（不触碰平台 PyTorch）：{need}")
        if not _pip_would_touch_torch(need):
            raise SystemExit("🔴 pip 安装计划里包含 torch/torchvision/torchaudio——执行会破坏平台 ROCm 环境，已拒绝安装。请把本 Cell 输出截图求助。")
        # 强制执行版本 spec（不走 pip_install 的"能import就跳过"检查——那会吞掉升级要求）
        last_err = ""
        for extra in ([], ["-i", "https://pypi.tuna.tsinghua.edu.cn/simple"]):
            r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *need],
                               capture_output=True, text=True)
            if r.returncode == 0:
                break
            last_err = r.stderr[-600:]
        else:
            raise SystemExit(f"🔴 上层依赖安装失败，请截图求助：\n{last_err}")
        post = _torch_state()
        if post != pre:
            raise SystemExit(f"🔴 依赖安装意外改变了平台 GPU 环境（{pre} → {post}），已停止以免损坏 ROCm。请截图求助。")
    import transformers, peft
    if _ver_tuple(transformers.__version__) < (5, 3) or _ver_tuple(peft.__version__) < (0, 20):
        raise SystemExit(f"🔴 安装后版本仍不满足要求（transformers {transformers.__version__} / peft {peft.__version__}，需要 ≥5.3 / ≥0.20）。请截图求助。")
    print(f"✅ 依赖就绪：torch {pre[0]}（HIP {pre[1]}）｜transformers {transformers.__version__}｜peft {peft.__version__}")

def _repo_pending_bytes(repo_id):
    """估算该仓库还需下载的字节数 = 仓库总大小 - 本地缓存已占（跳过符号链接避免重复计数）。
       用于磁盘门禁：已有完整缓存时不再受剩余空间限制，部分缓存时按实际待下量判断（保住断点续传）。"""
    try:
        info = HfApi(endpoint=CONFIG["hf_official"]).model_info(repo_id, files_metadata=True, timeout=30)
        total = sum((f.size or 0) for f in (info.siblings or []))
    except Exception:
        return None
    d = os.path.join(CONFIG["hf_cache"], "hub", "models--" + repo_id.replace("/", "--"))
    cached = 0
    if os.path.isdir(d):
        for root, _, fs in os.walk(d):
            for f in fs:
                p = os.path.join(root, f)
                if os.path.islink(p):
                    continue
                try:
                    cached += os.path.getsize(p)
                except OSError:
                    pass
    return max(total - cached, 0)

def _is_oom(e):
    s = (type(e).__name__ + " " + str(e)).lower()
    return "outofmemory" in s or "out of memory" in s or "cuda oom" in s

def _hint_old_4bit_cache():
    """旧 4bit 底模缓存（A10 时代遗留，约22GB）：只提示，绝不自动删除。"""
    d = os.path.join(CONFIG["hf_cache"], "hub", "models--unsloth--Qwen3.8-27B-unsloth-bnb-4bit")
    if os.path.isdir(d):
        print(f"💡 检测到旧的 4bit 底模缓存（约22GB，{d}）。当前 BF16 路线用不到它；")
        print("   确认不再需要 A10 4bit 路线后，可在 Terminal 手动清理（不清理也不影响运行）：")
        print(f'   rm -rf "{d}"')

# ---------- 模型②加载（AMD 主路线：BF16 全精度 + LoRA） ----------
def load_xuanhuan():
    cfg = CONFIG["models"]["xuanhuan"]
    STATE["smoke_ok"] = False   # 门禁默认关闭：只有本次加载后的 Smoke Test 全部通过才置 True
    ensure_bench_deps()
    # 磁盘门禁按"实际待下载量"判断：完整缓存直接放行、部分缓存保住断点续传
    pending = _repo_pending_bytes(cfg["base_repo"])
    if os.path.exists(CONFIG["workspace_root"]):
        _, _, free = shutil.disk_usage(CONFIG["workspace_root"])
        print(f"   （磁盘检查：剩余 {free/1000**3:.0f}GB｜BF16 底模本次预计还需下载 {pending/1000**3:.0f}GB）" if pending is not None
              else f"   （磁盘检查：剩余 {free/1000**3:.0f}GB｜待下载量估算失败，按保守规则判断）")
        if pending is not None and pending <= 0:
            pass   # 已有完整缓存，直接加载，不受剩余空间限制
        elif pending is not None:
            need = pending + 6 * 1000**3   # 安全余量
            if free < need:
                raise SystemExit(f"🔴 磁盘不足：本次还需下载约 {pending/1000**3:.0f}GB（建议预留 {need/1000**3:.0f}GB），当前剩余 {free/1000**3:.0f}GB。请清理 /mnt/workspace 后重试。")
        else:
            if free < 58 * 1000**3:   # 无法估算时的保守门槛
                raise SystemExit(f"🔴 磁盘不足（保守判断，剩余 {free/1000**3:.0f}GB）：BF16 底模约 55.6GB。请清理 /mnt/workspace 后重试。")
    print("📥 下载 BF16（未量化）底模 unsloth/Qwen3.8-27B（约55.6GB）+ LoRA 适配器（约1GB）…")
    base_dir = smart_download(cfg["base_repo"])
    adapter_dir = smart_download(cfg["adapter_repo"])
    _hint_old_4bit_cache()
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    print("⏳ 以 BF16 加载底模到 GPU（55.6GB 磁盘→显存，约5~15分钟，请勿中断）…")
    try:
        # 与作者官方用法一致：不量化、不offload、整卡直载
        base = AutoModelForCausalLM.from_pretrained(
            base_dir, torch_dtype=torch.bfloat16, device_map={"": 0}, low_cpu_mem_usage=True)
    except Exception as e:
        if _is_oom(e):
            raise SystemExit("🔴 在大显存 AMD 环境仍显存不足——这是环境/加载问题而非显存不够用。"
                             "请把下方原始报错截图求助（不做自动降级，避免掩盖根因）：\n" + str(e)[:500])
        raise
    # tokenizer 优先取 adapter 仓库（作者原版 tokenizer + chat template），失败才回退并大声提示
    try:
        tok = AutoTokenizer.from_pretrained(adapter_dir)
        tok_src = "adapter 仓库（作者原版）"
    except Exception as e:
        print(f"⚠️ adapter tokenizer 加载失败（{str(e)[:150]}）——已回退至 base tokenizer，本次结果与作者原测试环境存在差异！")
        tok = AutoTokenizer.from_pretrained(base_dir)
        tok_src = "base 仓库（回退）"
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    # 作者官方代码：pad token 缺失时用 eos 兜底
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    STATE.update(loaded_model="xuanhuan", runner="transformers", model=model, tokenizer=tok,
                 ctx=cfg["context"], offloaded=False, tokenizer_src=tok_src)
    show_gpu("模型②加载后")
    _smoke_test_xuanhuan()

def _smoke_test_xuanhuan():
    """加载成功后的技术自检：官方结构小样例 → 生成约96 token →
       检查 AMD 后端 / 模型驻留GPU / thinking 可关闭且未出现 / 输出正常。
       全部通过才放行正式测试；任一失败则阻止并说明原因（避免得出失真的 benchmark 结论）。"""
    import torch
    cfg = CONFIG["models"]["xuanhuan"]
    print("\n🧪 Smoke Test（约1~3分钟）：验证 AMD GPU、BF16 加载、思考关闭与生成是否全部正常…")
    results = []
    results.append(("AMD ROCm 后端（torch.version.hip）", bool(getattr(torch.version, "hip", None))))
    try:
        results.append((f"模型驻留GPU（保留显存 {torch.cuda.max_memory_reserved()/1024**3:.0f}GiB > 40GiB）",
                        torch.cuda.max_memory_reserved() > 40 * 1024**3))
    except Exception:
        results.append(("模型驻留GPU", False))
    thinking_ok = False
    gen = ""
    try:
        msgs = [{"role": "system", "content": cfg["system_prompt"]},
                {"role": "user", "content": SMOKE_PROMPT_USER}]
        tok = STATE["tokenizer"]
        try:
            prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                             enable_thinking=not cfg["disable_thinking"])
            thinking_ok = True
        except TypeError:
            thinking_ok = False
        results.append(("chat template 支持关闭 thinking", thinking_ok))
        if thinking_ok:
            inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to(STATE["model"].device)
            t0 = time.time()
            with torch.inference_mode():
                out = STATE["model"].generate(**inputs, max_new_tokens=96, do_sample=True,
                                              temperature=cfg["temperature"], top_p=cfg["top_p"],
                                              top_k=cfg["top_k"], repetition_penalty=cfg["repetition_penalty"],
                                              use_cache=True)
            gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            dt = time.time() - t0
            print(f"   （Smoke 样例生成 {96/max(dt,1e-6):.1f} tokens/s｜显存峰值 {torch.cuda.max_memory_allocated()/1024**3:.1f}GiB）")
    except Exception as e:
        print(f"   生成异常：{type(e).__name__}: {str(e)[:200]}")
    # 非空 <think> 段落 = thinking 没关住；NaN/空输出 = 数值或加载异常
    has_think = bool(re.search(r"<think>\s*\S", gen))
    ok_gen = bool(gen.strip()) and not has_think and "nan" not in gen.lower()
    results.append(("生成成功且无思考内容/NaN", ok_gen))
    for name, v in results:
        print(f"   {'✅' if v else '❌'} {name}")
    if all(v for _, v in results):
        print("✅ 玄幻 DPO 已在 AMD ROCm GPU 上准备完成（BF16 + LoRA + thinking 关闭），可以进行正式小说测试！")
        STATE["smoke_ok"] = True
    else:
        STATE["smoke_ok"] = False
        print("🔴 Smoke Test 未通过：正式测试已被阻止（避免得出失真的 benchmark 结论）。请把上面的 ❌ 项截图求助。")

# ---------- 模型①加载（llama.cpp 路线，面向 NVIDIA；AMD 适配属第二阶段） ----------
def ensure_llama_cpp():
    """按优先级准备 llama.cpp：
       ① CUDA 预编译 pip 轮子（约1~2分钟，可流式输出）
       ② llama.cpp 官方预编译二进制（含 CUDA 的 Ubuntu 包）
       ③ 源码编译（首次约15~30分钟，仅需一次）"""
    try:
        import llama_cpp
        print(f"✅ llama-cpp-python 已就绪（{getattr(llama_cpp, '__version__', '?')}）")
        return "python"
    except ImportError:
        pass
    v = _run("nvidia-smi | grep -o 'CUDA Version: [0-9.]*'")
    m = re.search(r"([0-9]+)\.([0-9]+)", v or "")
    cus = []
    if m:
        cus.append(f"cu{m.group(1)}{m.group(2)}")
    cus += ["cu126", "cu125", "cu124", "cu121"]
    for cu in dict.fromkeys(cus):
        print(f"⏳ 安装 llama-cpp-python（{cu} CUDA 预编译版，约1~2分钟）…")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python",
                        "--extra-index-url", f"https://abetlen.github.io/llama-cpp-python/whl/{cu}"],
                       capture_output=True, text=True)
        try:
            import llama_cpp
            print(f"✅ 安装成功（{cu}）")
            return "python"
        except ImportError:
            continue
    try:
        import requests
        rel = requests.get("https://api.github.com/repos/ggml-org/llama.cpp/releases/latest", timeout=30).json()
        assets = [a for a in rel.get("assets", []) if "ubuntu" in a["name"].lower() and a["name"].endswith(".zip")]
        cuda_assets = [a for a in assets if "cuda" in a["name"].lower() or "cublas" in a["name"].lower()]
        pick = (cuda_assets or assets)[0]
        print(f"⏳ pip 轮子不可用，下载官方预编译二进制：{pick['name']} …")
        import urllib.request
        zpath = os.path.join(CONFIG["models_root"], pick["name"])
        urllib.request.urlretrieve(pick["browser_download_url"], zpath)
        bdir = os.path.join(CONFIG["models_root"], "llama-bin")
        with zipfile.ZipFile(zpath) as z:
            z.extractall(bdir)
        cli = glob.glob(os.path.join(bdir, "**", "llama-cli*"), recursive=True)
        STATE["llama_bin"] = cli[0] if cli else None
        if STATE.get("llama_bin"):
            print(f"✅ 官方二进制就绪：{STATE['llama_bin']}（此路线为非流式输出）")
            return "binary"
    except Exception as e:
        print(f"   · 官方二进制不可用（{str(e)[:120]}），转为源码编译…")
    print("⏳ 源码编译 llama.cpp（首次约 15~30 分钟，仅需一次，请耐心等待）…")
    env = dict(os.environ, CMAKE_ARGS="-DGGML_CUDA=on", FORCE_CMAKE="1")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "llama-cpp-python"],
                       env=env)
    if r.returncode != 0:
        raise RuntimeError("llama.cpp 三种安装方式都失败了，请截图上面的报错信息求助。")
    print("✅ 源码编译完成")
    return "python"

def load_webnovel():
    if GPU_KIND == "amd":
        print("ℹ️  模型①（llama.cpp 路线）当前面向 NVIDIA GPU，AMD 适配属于第二阶段。")
        print("    本阶段请使用模型②（玄幻 DPO，默认）。A10/4bit 历史方案见 main 分支。")
        return
    cfg = CONFIG["models"]["webnovel"]
    print(f"📥 模型①：检查/下载 {cfg['repo_id']}（{cfg['quant']}，约16.5GB；下载过则直接复用）")
    gguf_path = smart_download(cfg["repo_id"], cfg["gguf_file"])
    runner = ensure_llama_cpp()
    if runner == "python":
        from llama_cpp import Llama
        before = _run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits").strip()
        print("⏳ 加载到 GPU（约1~3分钟）…")
        llm = Llama(model_path=gguf_path, n_ctx=cfg["context"], n_gpu_layers=-1,
                    n_threads=max((os.cpu_count() or 8) // 2, 2), verbose=False)
        after = _run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits").strip()
        try:
            delta = float(after) - float(before)
        except Exception:
            delta = 0
        if delta < 4000:   # MiB：显存没涨说明没进GPU，会极慢
            print("⚠️ 检测到模型可能没有加载进 GPU（显存占用未上升）。可继续，但生成速度会很慢。")
        STATE.update(loaded_model="webnovel", runner="python", llm=llm, ctx=cfg["context"], offloaded=False)
    else:
        STATE.update(loaded_model="webnovel", runner="binary", llm=gguf_path, ctx=cfg["context"], offloaded=False)
    show_gpu("模型①加载后")
    print("✅ 模型①就绪！请到第 6 步粘贴剧情、点【开始生成】。")

# ---------- 统一的加载/卸载入口 ----------
def unload_current():
    if STATE.get("loaded_model") is None:
        return
    for k in ("llm", "model", "tokenizer", "smoke_ok"):   # smoke_ok 必须一并清掉，防止旧状态污染
        STATE.pop(k, None)
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass
    STATE.update(loaded_model=None, runner=None)
    print("🧹 已释放上一个模型的显存")

def load_model(key):
    if NO_GPU:
        print("⚠️ 当前是 CPU 环境：只用于预下载模型，不能加载/生成。请切换到 AMD GPU 实例后重新运行。")
        return
    if STATE.get("loaded_model") == key:
        print("✅ 该模型已在内存中，无需重复加载。")
        return
    unload_current()
    if CONFIG["models"][key]["type"] == "gguf":
        load_webnovel()
    else:
        load_xuanhuan()

# ---------- 交互控件 ----------
import ipywidgets as widgets
from IPython.display import display, clear_output

_sel = widgets.Dropdown(options=[(v["label"], k) for k, v in CONFIG["models"].items()],
                        value=CONFIG["default_model"], description="选择模型：",
                        layout=widgets.Layout(width="auto"))
_btn = widgets.Button(description="加载 / 切换模型", button_style="primary")
_out = widgets.Output()
def _on_load(b):
    with _out:
        clear_output()
        try:
            load_model(_sel.value)
        except SystemExit as e:
            print(str(e))
        except Exception as e:
            print(f"🔴 加载失败：{type(e).__name__}: {str(e)[:300]}\n   常见原因：网络中断（重跑可续传）/ 磁盘不足（清理 /mnt/workspace）。")
_btn.on_click(_on_load)
display(widgets.VBox([widgets.HBox([_sel, _btn]), _out]))

# 「运行全部」时：GPU 环境自动加载默认模型②；CPU 环境改为预下载（不占 GPU 时长）
if NO_GPU:
    print("📥 CPU 预下载模式：开始下载玄幻 DPO 所需文件（BF16 底模 55.6GB + LoRA 约1GB，CPU 实例时长免费不限）…")
    print("   注意：本阶段不下载模型①（WebNovel Writer，第二优先级）。")
    _c = CONFIG["models"]["xuanhuan"]
    smart_download(_c["base_repo"])
    smart_download(_c["adapter_repo"])
    print("\n✅ 预下载完成！模型已永久保存在 /mnt/workspace/models（关机不丢）。")
    print("   下一步：停止本 CPU 实例 → 启动 AMD GPU 实例（镜像 ubuntu22.04-rocm7.2.3-py312-torch2.11.0-1.39.0）→")
    print("         重新 Run All → 模型从磁盘加载 → 自动 Smoke Test → 第 6 步生成正文。")
else:
    load_model(CONFIG["default_model"])

## 第 6 步 · 粘贴剧情提示词 → 生成正文

输入框已预填**作者官方 〖〗结构示例**（人物状态 / 前情 / 事件骨架 / 写作要求）——把示例换成你的剧情即可；直接粘贴纯剧情（骨架）也可以，程序会自动套进官方结构。点【✍️ 开始生成】后正文逐字流式显示，完自动保存到 `results/`。

生成参数使用作者 held-out 测试原参数（temperature=0.85 / top_p=0.90 / top_k=40 / 重复惩罚=1.05，上限 1700 token），篇幅由提示词控制在约 500~1000 字。

In [ ]:
# ============================================================
# 第 6 步：生成界面（官方结构输入 + 流式输出 + 自动保存 + Smoke 门禁）
# ============================================================
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

def _default_input(key):
    if key == "webnovel":
        return ("（示例，请替换成你的剧情）\n"
                "少年在废弃藏经阁扫地时捡到半页残缺剑谱，当夜梦见剑招自行演练。\n"
                "醒来后他试着比划，被巡夜长老撞见。")
    return ("〖人物状态〗\n（示例）沈砚：外门杂役弟子，炼气九层，性情谨慎，掌心藏有黑铁。\n\n"
            "〖前情〗\n（示例）沈砚在坊市拍卖会上与内门弟子赵崂结怨，对方撂话三日后演武场见。\n\n"
            "〖事件骨架〗\n（示例）\n1. 沈砚后山寒潭淬体。\n2. 黑铁吸食气血，反哺先天罡气。\n3. 突破至筑基。\n\n"
            "〖写作要求〗写成完整连续的小说场景。\n\n——把示例换成你的剧情；保留〖〗结构只改内容效果最好，直接粘贴纯骨架也可以。")

def run_generation(model_key, user_msg, system_prompt, cb=None):
    """统一生成入口。cb 为逐段回调（用于流式显示）。返回 dict(text, elapsed, gen)。
       模型②必须 Smoke Test 明确通过（is True）才放行——None/False/旧状态一律禁止。"""
    if STATE.get("loaded_model") != model_key:
        load_model(model_key)
    if model_key == "xuanhuan" and STATE.get("smoke_ok") is not True:
        raise RuntimeError("⛔ Smoke Test 未通过或未完成，正式生成已被阻止（保证 benchmark 有效性）。请回第 5 步查看 ❌ 原因并重新加载模型。")
    t0 = time.time()
    if model_key == "webnovel":
        text, ginfo = _gen_webnovel(system_prompt, user_msg, cb)
    else:
        text, ginfo = _gen_xuanhuan(system_prompt, user_msg, cb)
    return {"text": text, "elapsed": time.time() - t0, "gen": ginfo}

def _gen_webnovel(sys_p, user_p, cb):
    cfg = CONFIG["models"]["webnovel"]
    msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": user_p}]
    if STATE["runner"] == "python":
        try:
            stream = STATE["llm"].create_chat_completion(
                msgs, temperature=cfg["temperature"], top_p=cfg["top_p"],
                repeat_penalty=cfg["repeat_penalty"], max_tokens=cfg["max_new_tokens"],
                stream=True, jinja=True)   # jinja=True：使用 GGUF 内置的官方对话模板
            parts = []
            for ch in stream:
                d = ch["choices"][0]["delta"].get("content", "")
                if d:
                    parts.append(d)
                    if cb: cb(d)
            return clean_output("".join(parts)), {}
        except TypeError:
            pass   # 旧版本不支持 jinja 参数 → 手工套 Qwen 对话格式
        prompt = (f"<|im_start|>system\n{sys_p}<|im_end|>\n"
                  f"<|im_start|>user\n{user_p}<|im_end|>\n<|im_start|>assistant\n")
        stream = STATE["llm"](prompt=prompt, temperature=cfg["temperature"], top_p=cfg["top_p"],
                              repeat_penalty=cfg["repeat_penalty"], max_tokens=cfg["max_new_tokens"],
                              stream=True)
        parts = []
        for ch in stream:
            d = ch["choices"][0].get("text", "")
            if d:
                parts.append(d)
                if cb: cb(d)
        return clean_output("".join(parts)), {}
    # binary 兜底路线：llama-cli 非流式，一次输出全文
    import tempfile, subprocess
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8") as tf:
        tf.write(f"<|im_start|>system\n{sys_p}<|im_end|>\n<|im_start|>user\n{user_p}<|im_end|>\n<|im_start|>assistant\n")
        pf = tf.name
    cmd = [STATE["llama_bin"], "-m", STATE["llm"], "-f", pf, "-c", str(cfg["context"]),
           "-n", str(cfg["max_new_tokens"]), "-ngl", "-1", "--temp", str(cfg["temperature"]),
           "--top-p", str(cfg["top_p"]), "--top-k", "40", "--repeat-penalty", str(cfg["repeat_penalty"]),
           "--no-display-prompt"]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    return clean_output(r.stdout.strip()), {}

def _gen_xuanhuan(sys_p, user_p, cb):
    import torch
    from transformers import TextIteratorStreamer
    from threading import Thread
    cfg = CONFIG["models"]["xuanhuan"]
    tok, model = STATE["tokenizer"], STATE["model"]
    msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": user_p}]
    # 严格关闭思考：模板不支持 enable_thinking 时直接报错，绝不静默继续（否则 benchmark 失真）
    try:
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=not cfg["disable_thinking"])
    except TypeError:
        raise RuntimeError("当前 transformers / chat template 不支持 enable_thinking=False，无法关闭思考模式——"
                           "正式测试会失真。请确认第 5 步已将 transformers 升到 ≥5.3 后重新加载模型。")
    # ---- 4096 上下文真正作为门禁（作者训练分布：max_seq 4096 / max_prompt 2800 / max_completion 1900）----
    ptoks = len(tok(prompt, add_special_tokens=False)["input_ids"])
    if ptoks > cfg.get("max_prompt_tokens", 2800):
        raise RuntimeError(f"⚠️ Prompt 过长：{ptoks} tokens，超过作者训练分布的提示词上限（2800）。"
                           "标准 benchmark 不执行——请缩短剧情/前情后重试（程序不会自动截断你的剧情）。")
    actual_max_new = min(cfg["max_new_tokens"], cfg["context"] - ptoks)
    if actual_max_new < 200:
        raise RuntimeError(f"⚠️ 剩余生成空间不足：Prompt 已用 {ptoks}/{cfg['context']} tokens，最多只能再生成 {actual_max_new} tokens。"
                           "请缩短 Prompt。")
    print(f"📏 Prompt {ptoks} tokens｜本次生成上限 {actual_max_new}｜合计 {ptoks + actual_max_new} / {cfg['context']} tokens ✅")
    inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    kw = dict(**inputs, max_new_tokens=actual_max_new, do_sample=True,
              temperature=cfg["temperature"], top_p=cfg["top_p"], top_k=cfg["top_k"],
              repetition_penalty=cfg["repetition_penalty"], use_cache=True, streamer=streamer)

    def _worker():
        with torch.inference_mode():
            model.generate(**kw)
    t = Thread(target=_worker)
    t.start()
    parts = []
    for d in streamer:
        if d:
            parts.append(d)
            if cb: cb(d)
    t.join()
    raw = "".join(parts)
    # 思考内容出现 = thinking 没关住：本次结果作废，绝不清洗后伪装成正常 benchmark 正文
    if "<think>" in raw:
        raise RuntimeError("⛔ 本次生成出现 <think> 思考内容——thinking 未真正关闭，本次结果作废、不计入 benchmark（已拦截，未保存）。请截图反馈。")
    return raw.strip(), {"prompt_tokens": ptoks, "actual_max_new_tokens": actual_max_new}

# ---------- 界面 ----------
_cur = lambda: STATE.get("loaded_model") or CONFIG["default_model"]
_task_w = widgets.Dropdown(options=list(WEBNOVEL_TASKS.keys()), value=WEBNOVEL_DEFAULT_TASK,
                           description="任务类型：", layout=widgets.Layout(width="auto"),
                           tooltip="仅模型①使用：决定用模型卡里的哪种官方模板包装你的剧情")
_text_w = widgets.Textarea(value=_default_input(_cur()), placeholder="把你的剧情粘贴到这里…",
                           layout=widgets.Layout(width="100%", height="240px"))
_tpl_btn = widgets.Button(description="填入该模型推荐模板", button_style="info",
                          tooltip="清空输入框并填入当前模型的推荐输入格式")
_gen_btn = widgets.Button(description="✍️ 开始生成", button_style="success")
_gen_out = widgets.Output()

def _on_tpl(b):
    _text_w.value = _default_input(_cur())
_tpl_btn.on_click(_on_tpl)

def _on_gen(b):
    with _gen_out:
        clear_output()
        if NO_GPU:
            print("⚠️ 当前是 CPU 环境，无法生成正文。请按第 5 步的提示切换到 AMD GPU 实例。")
            return
        key = _cur()
        if key == "xuanhuan" and STATE.get("loaded_model") == "xuanhuan" and STATE.get("smoke_ok") is not True:
            print("⛔ 模型②的 Smoke Test 未通过或未完成，正式测试已被阻止。请回到第 5 步查看 ❌ 原因（或重新运行加载）。")
            return
        task = _task_w.value if key == "webnovel" else "官方结构→正文"
        sys_p = CONFIG["models"][key]["system_prompt"]
        try:
            user_msg = build_user_message(key, _text_w.value, _task_w.value)
        except ValueError as e:
            print(f"⚠️ {e}")
            return
        print(f"🚀 模型：{SHORT_NAME[key]}｜任务：{task}｜参数：作者官方推荐值（详见保存的 .md 记录）")
        print(f"⏳ 生成中（目标 {CONFIG['target_chars']} 字，预计1~5分钟），正文如下 ↓\n" + "─" * 60)
        t0 = time.time()
        try:
            res = run_generation(key, user_msg, sys_p, cb=lambda d: print(d, end="", flush=True))
            dt = time.time() - t0
            text = res["text"]
            print("\n" + "─" * 60)
            print(f"✍️ 共 {len(text)} 字，用时 {dt:.0f} 秒（{len(text)/max(dt,1):.1f} 字/秒）")
            try:
                import torch
                peak = f"{torch.cuda.max_memory_allocated()/1024**3:.1f}GiB"
            except Exception:
                peak = "?"
            meta = {"elapsed": f"{dt:.0f}", "speed": f"{len(text)/max(dt,1):.1f} 字/秒",
                    "vram": _run("amd-smi --brief | head -2 || nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader").strip() or "见峰值",
                    "vram_peak": peak, "context": STATE.get("ctx", CONFIG["models"][key].get("context")),
                    **(res.get("gen") or {})}
            save_result(key, task, _text_w.value, text, meta)
            show_gpu("生成后")
        except KeyboardInterrupt:
            print("\n⏹️ 已手动停止。调整输入后可再次点击生成。")
        except SystemExit as e:
            print(str(e))
        except Exception as e:
            print(f"\n🔴 生成失败：{type(e).__name__}: {str(e)[:300]}")
            print("   排查建议：① 回第 5 步重新加载模型；② 把报错截图求助，不要自行改代码。")

_gen_btn.on_click(_on_gen)
display(widgets.VBox([_text_w, widgets.HBox([_task_w, _tpl_btn, _gen_btn]), _gen_out]))
print("👆 在输入框粘贴剧情（或点【填入该模型推荐模板】），然后点【开始生成】。结果自动保存到 results/ 文件夹。")

## 第 7 步 · 换另一个模型再测（对比）

回到**第 5 步**：下拉框选另一个模型 → 点【加载 / 切换模型】→ 回到**第 6 步**粘贴同一段剧情再生成。

> ⚠️ 当前分支（AMD 主路线）下，模型①（WebNovel Writer，llama.cpp）面向 NVIDIA GPU，在 AMD 实例上会提示跳过——它的 AMD 适配属于第二阶段。两条模型的对比将在模型①适配完成后恢复。

## 第 8 步 ·（可选）批量盲评模式

把若干测试剧情放进 Notebook 同级的 `prompts/` 文件夹（一个 `.txt` 一条，示例会自动生成），把 `BATCH_MODELS` 改成 `["xuanhuan"]` 并重新运行本 Cell。输出保存在 `results/anonymous/`，文件名**隐藏模型名**（`sample_001_A.txt`…），对照表在「AAA_评分后再看_对照表.txt」——**评分前别打开**。

> 当前 AMD 分支只跑模型②；等模型①适配 AMD 后可改 `["webnovel", "xuanhuan"]` 做双模型盲评。

In [ ]:
# ============================================================
# 第 8 步：批量盲评模式（读取 prompts/ 文件夹 → 逐个生成 → 匿名保存）
# ============================================================
BATCH_MODELS = []        # ← 启用请改成 ["xuanhuan"]（模型①AMD适配完成后可加 "webnovel"）

if not BATCH_MODELS:
    print('ℹ️  批量模式未启用。把上方 BATCH_MODELS 改成 ["xuanhuan"] 后重新运行本 Cell。')
    print("   提示：prompts/ 文件夹放测试剧情（.txt），首次运行会自动生成 3 个示例。")
else:
    files = ensure_prompts_dir()
    if not files:
        print("🔴 prompts/ 里没有 .txt 文件，请先放入测试剧情。")
    else:
        letters = {k: chr(65 + i) for i, k in enumerate(CONFIG["models"])}   # 按配置顺序 A/B
        anon_dir = os.path.join(CONFIG["results_dir"], "anonymous")
        os.makedirs(anon_dir, exist_ok=True)
        run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        key_path = os.path.join(anon_dir, "AAA_评分后再看_对照表.txt")
        mapping_lines = [f"盲评运行 {run_ts} 对照表（评分前请勿查看！）\n"]
        print(f"▶ 批量模式：{len(files)} 个提示词 × {len(BATCH_MODELS)} 个模型 = {len(files)*len(BATCH_MODELS)} 次生成\n")
        for key in BATCH_MODELS:
            if key == "webnovel" and GPU_KIND == "amd":
                print(f"⏭️  跳过 {SHORT_NAME['webnovel']}：llama.cpp 路线暂未适配 AMD（第二阶段）。")
                continue
            load_model(key)
            if STATE.get("loaded_model") != key:
                print(f"⏭️  {key} 未能加载，跳过。")
                continue
            mapping_lines.append(f"{letters[key]} = {SHORT_NAME[key]}（{_repo_line(key)}）")
            for i, pf in enumerate(files, 1):
                with open(pf, encoding="utf-8") as f:
                    raw = f.read()
                task = WEBNOVEL_DEFAULT_TASK if key == "webnovel" else "官方结构→正文"
                user_msg = build_user_message(key, raw, task)
                print(f"  [{SHORT_NAME[key]}] {i}/{len(files)} {os.path.basename(pf)} 生成中…", end=" ", flush=True)
                t0 = time.time()
                res = run_generation(key, user_msg, CONFIG["models"][key]["system_prompt"])
                dt = time.time() - t0
                meta = {"elapsed": f"{dt:.0f}", "speed": f"{len(res['text'])/max(dt,1):.1f} 字/秒",
                        "vram": "", "context": STATE.get("ctx"), **(res.get("gen") or {})}
                save_result(key, task, raw, res["text"], meta)
                anon = os.path.join(anon_dir, f"sample_{i:03d}_{letters[key]}.txt")
                with open(anon, "w", encoding="utf-8") as f:
                    f.write(res["text"].strip() + "\n")
                print(f"完成（{dt:.0f}秒）→ {os.path.basename(anon)}")
            unload_current()
        with open(key_path, "a", encoding="utf-8") as f:
            f.write("\n".join(mapping_lines) + "\n\n")
        print(f"\n✅ 批量完成。匿名结果在 {anon_dir}\n   对照表（评分后再看）：{key_path}")

## 显存随时查看（小工具）

任何时候想看显存占用，运行下面这个 Cell 即可（AMD / NVIDIA 通用）。

In [ ]:
# 显存实时查看
show_gpu("实时")

## 🎬 用完之后：关闭 GPU 实例（省时长）

- AMD GPU 免费额度约 **100 小时**，按**开机时间**计算（下载期间也在计时）——用完就停止实例；
- 停止路径：ModelScope 网页 →「我的Notebook」→ 对应实例 → **停止**；
- `/mnt/workspace` 里的模型和结果**全部保留**（100GB 持久盘），下次启动直接复用、不重新下载；
- 只有**删除实例**才会清空文件。删除/长期保存前，把 `results/` 里的正文下载到本地：Notebook 左侧文件列表 → 右键 → **Download**。

---

**许可与用途提醒（详见仓库 README）**：本项目仅用于个人对比测试。两个模型分别遵循各自模型卡的许可条款；模型②的训练语料含受版权保护文学作品的派生材料，适配器不授予对底层文学作品的任何权利。任何正式或商业使用前，请分别审查两个模型的许可与训练数据权利。